# Lab 5: Talwani Forward Modeling and Linear Inversion

| | |
|---|---|
| **Module** | M5 — Forward and Inverse Modeling |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M5-1, M5-2; Homework 5, Problems 5.1–5.2; Lab 2 |
| **Textbook** | Blakely Ch. 9, 10 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Implement** the Talwani (1959) algorithm for the gravity of a 2D polygonal body
2. **Construct** the sensitivity matrix **G** for a linear gravity inverse problem
3. **Solve** an underdetermined inverse problem with Tikhonov regularization and interpret the trade-off
4. **Apply** Euler deconvolution to estimate source depth from a total-field magnetic anomaly

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely

**Your response:** cells require written answers. Hints: `print(hints['key'])`.


## Background

### Talwani's method

The **Talwani method** (Talwani et al. 1959; Blakely §9.2) computes the vertical
gravitational attraction of a 2D body with arbitrary cross-section, approximated
as a polygon.  For a polygon with $N$ vertices $(x_i, z_i)$, the attraction at
a surface point $x_0$ is

$$
g_z = 2G\Delta\rho \sum_{i=1}^{N} \left[
  z_{i+1}\,\theta_{i+1} - z_i\,\theta_i
  + \frac{x_{i+1}-x_i}{z_{i+1}-z_i}\,(z_{i+1}\cos\theta_{i+1} - z_i\cos\theta_i
    - r_{i+1}\sin\theta_{i+1} + r_i\sin\theta_i)
\right] \cdot \ldots
$$

where $\theta_i = \arctan\bigl((z_i)/(x_i - x_0)\bigr)$ and
$r_i = \sqrt{(x_i - x_0)^2 + z_i^2}$.  The standard Blakely formulation
(eq. 9.19) is provided in the implementation scaffold below.

### Linear inversion

For a fixed geometry, the gravity is **linear** in density:

$$
\mathbf{d} = \mathbf{G}\,\mathbf{m} + \mathbf{n}
$$

where $\mathbf{d}$ is the data vector (observed gravity), $\mathbf{m}$ is the
model vector (densities of cells), and $\mathbf{G}$ is the sensitivity matrix.
We solve the Tikhonov-regularized least-squares problem:

$$
\hat{\mathbf{m}} = \arg\min_\mathbf{m}\left\|
  \mathbf{G}\mathbf{m} - \mathbf{d}
\right\|^2 + \lambda^2\|\mathbf{m}\|^2
$$

which has the closed-form solution
$\hat{\mathbf{m}} = (\mathbf{G}^T\mathbf{G} + \lambda^2 \mathbf{I})^{-1}\mathbf{G}^T\mathbf{d}$.

### Euler deconvolution

**Euler's homogeneity relation** (Blakely §10.3) states that for a potential
field $F$ from a source at $(x_0, z_0)$:

$$
(x - x_0)\frac{\partial F}{\partial x} + (z - z_0)\frac{\partial F}{\partial z} + N\,F = 0
$$

where $N$ is the **structural index** (0 = contact, 1 = dyke/pipe, 2 = sphere).
Solving over a sliding window yields an ensemble of $(x_0, z_0)$ estimates
that cluster near the true source.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

G_grav = 6.674e-11   # gravitational constant, N m^2 kg^-2

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})
print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_talwani_loop': (
        "Loop over each edge (i -> i+1) of the polygon (wrap index with % N_verts).\n"
        "For each edge compute:\n"
        "  theta_i = arctan2(z_i, x_i - x_obs)   [note: arctan2 not arctan]\n"
        "  r_i     = sqrt((x_i - x_obs)^2 + z_i^2)\n"
        "Then apply the Talwani segment formula (Blakely eq. 9.19).\n"
        "Watch out for the degenerate case where z_{i+1} == z_i."
    ),
    'p1_talwani_formula': (
        "Blakely eq. 9.19 per edge from vertex i to i+1:\n"
        "  x1 = x_i - x_obs,  z1 = z_i\n"
        "  x2 = x_{i+1} - x_obs, z2 = z_{i+1}\n"
        "  alpha = (z2 - z1) / (x2 - x1)  [slope of edge; handle vertical edges separately]\n"
        "  beta  = z1 - alpha * x1\n"
        "  t1 = arctan2(z1, x1),  t2 = arctan2(z2, x2)\n"
        "  r1 = sqrt(x1^2+z1^2),  r2 = sqrt(x2^2+z2^2)\n"
        "  factor = alpha^2 / (1+alpha^2)\n"
        "  contrib = alpha*(t2-t1) + log(r2/r1)*(beta*alpha/(alpha^2+1))\n"
        "           ... actually use the formulation in Blakely directly."
    ),
    'p2_G_matrix': (
        "Each column j of G corresponds to a subsurface cell.\n"
        "Each row i corresponds to a surface observation point.\n"
        "G[i,j] = gz contribution at station i from cell j with unit density contrast.\n"
        "Use talwani_gz applied to the rectangle polygon of cell j, with rho=1."
    ),
    'p2_tikhonov': (
        "Tikhonov solution: m_hat = inv(G^T G + lam^2 * I) @ G^T @ d\n"
        "Use scipy.linalg.solve or np.linalg.lstsq.\n"
        "Start with lam=0 (pure least-squares) and increase until the solution\n"
        "looks physically reasonable (smooth, bounded density contrasts)."
    ),
    'p3_euler_setup': (
        "Euler's equation for each window point x_k (sliding window of width W):\n"
        "  (x_k - x0)*dF/dx(x_k) + (z_k - z0)*dF/dz(x_k) + N*F(x_k) = 0\n"
        "Rearranged as a linear system [dF/dx, dF/dz, N]^T * [x0, z0, B]^T = RHS\n"
        "where B = N*B_base is a background term.\n"
        "Compute dF/dx numerically (np.gradient) before building the system."
    ),
}
# print(hints['p1_talwani_loop'])

---
## Part 1: Talwani Forward Modeling *(Guided — ~45 min)*

We implement the Talwani method for 2D polygonal bodies, verify it against
the analytic sphere formula from Lab 2, then use it to model a geologically
realistic cross-section.

**Available hints:** `p1_talwani_loop`, `p1_talwani_formula`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Talwani algorithm for a 2D polygon.

def talwani_gz(x_obs, poly_x, poly_z, delta_rho):
    """
    Vertical gravitational attraction of an infinite 2D polygonal body.

    Uses the Talwani et al. (1959) line-integral formula (Blakely eq. 9.19).

    Parameters
    ----------
    x_obs : ndarray, shape (M,)
        Observation positions along the surface profile, meters.
    poly_x : ndarray, shape (N,)
        x-coordinates of polygon vertices, meters.
    poly_z : ndarray, shape (N,)
        z-coordinates (positive downward) of polygon vertices, meters.
    delta_rho : float
        Density contrast, kg/m^3.

    Returns
    -------
    gz : ndarray, shape (M,)
        Vertical attraction, m/s^2 (positive downward).

    Notes
    -----
    Vertices should be in clockwise order when viewed with z positive downward.
    The polygon is automatically closed (last vertex connected to first).

    For each observation point, sum contributions from each edge using
    Blakely eq. 9.19.  The formula for a single edge (x1,z1)->(x2,z2)
    in coordinates relative to the observation point is:

        theta1 = arctan2(z1, x1),  theta2 = arctan2(z2, x2)
        r1 = sqrt(x1^2+z1^2),      r2 = sqrt(x2^2+z2^2)

    If z1 != z2 (non-horizontal edge):
        phi = (x2-x1)/(z2-z1)   [reciprocal slope]
        edge_contrib = z2*(theta2 - theta1)*phi
                       + z2*log(r2/r1)*phi^2/(1+phi^2)
                       + (theta2 - theta1)/(1+phi^2)
                       - phi*log(r2/r1)/(1+phi^2)
        (see Blakely eq. 9.19 — verify sign carefully)

    If z1 == z2 (horizontal edge):
        edge_contrib = 0  (no vertical contribution from horizontal segment)

    gz_total = 2 * G * delta_rho * sum(edge_contrib)
    """
    x_obs  = np.asarray(x_obs, dtype=float)
    poly_x = np.asarray(poly_x, dtype=float)
    poly_z = np.asarray(poly_z, dtype=float)
    N_verts = len(poly_x)
    gz = np.zeros_like(x_obs)

    for obs_idx in range(len(x_obs)):
        xo = x_obs[obs_idx]
        total = 0.0

        for i in range(N_verts):
            j = (i + 1) % N_verts   # next vertex (wraps around)

            # Shift to observation-point-centered coordinates
            x1 = poly_x[i] - xo
            z1 = poly_z[i]
            x2 = poly_x[j] - xo
            z2 = poly_z[j]

            if np.isclose(z1, z2):
                continue   # horizontal edge: zero contribution

            # Step 1: angles and distances
            # YOUR CODE HERE
            raise NotImplementedError

            # Step 2: reciprocal slope phi = (x2-x1)/(z2-z1)
            # YOUR CODE HERE
            raise NotImplementedError

            # Step 3: Talwani segment formula (Blakely eq. 9.19)
            # YOUR CODE HERE
            raise NotImplementedError

            total += edge_contrib

        gz[obs_idx] = 2 * G_grav * delta_rho * total

    return gz

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────
# Test Talwani against the analytic sphere (approximated as a polygon).

def _check(label, got, expected, rtol=1e-3, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

# Build a fine circle polygon approximating a sphere cross-section
_R      = 500.0    # radius, m
_depth  = 2000.0   # depth to center, m
_drho   = 500.0    # density contrast, kg/m^3
_N_circ = 360      # number of polygon vertices
_angles = np.linspace(0, 2*np.pi, _N_circ, endpoint=False)
_cx = _R * np.cos(_angles)      # clockwise: flip sign
_cz = _depth + _R * np.sin(_angles)  # z positive downward
# Ensure clockwise order (z positive down: start at bottom, go clockwise)
_cx = _R * np.cos(-_angles)
_cz = _depth + _R * np.sin(-_angles)

# Analytic answer at x=0 and x=3000 m
_M = (4/3)*np.pi*_R**3*_drho
_x_test = np.array([0.0, 3000.0, -3000.0])
_gz_analytic = G_grav * _M * _depth / (_x_test**2 + _depth**2)**1.5

_gz_talwani = talwani_gz(_x_test, _cx, _cz, _drho)
_check('Talwani circle ≈ analytic sphere (1% tolerance)', _gz_talwani, _gz_analytic, rtol=0.01)

# Test symmetry: gz should be symmetric about x=0
_x_sym = np.linspace(-5000, 5000, 101)
_gz_sym = talwani_gz(_x_sym, _cx, _cz, _drho)
if np.allclose(_gz_sym, _gz_sym[::-1], rtol=1e-8):
    print('  ✓ PASS  Profile is symmetric about x=0')
else:
    print('  ✗ FAIL  Profile should be symmetric')

# Test linearity: doubling density doubles gz
_gz1 = talwani_gz(np.array([0.0]), _cx, _cz, _drho)
_gz2 = talwani_gz(np.array([0.0]), _cx, _cz, 2*_drho)
_check('Doubling density doubles gz', _gz2[0], 2*_gz1[0], rtol=1e-10)

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Geological scenario: a profile across a sedimentary basin.
#
# The basin is a graben — a down-dropped block bounded by two normal faults.
# The basin fill has lower density than the surrounding basement.
#
#  Surface: z = 0 km
#  Basin (low density):  trapezoid shape, wider at top, narrower at base
#  Basement (reference): everything else

# Profile
x_prof = np.linspace(-20e3, 20e3, 201)   # m, surface profile

# Basin polygon (vertices in clockwise order, z positive downward)
basin_x = np.array([-8e3, -5e3,  5e3,  8e3]) * 1.0   # m
basin_z = np.array([  0,   4e3,  4e3,    0]) * 1.0   # m (z positive down)
basin_drho = -400.0   # kg/m^3 (basin fill lighter than basement)

print('Geological model defined.')
print(f'Basin width at surface: {(basin_x[-1]-basin_x[0])/1000:.0f} km')
print(f'Basin depth: {basin_z.max()/1000:.0f} km')

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Compute and plot the Bouguer anomaly over the basin.
#
# Requirements:
#   - Top panel: cross-section showing basin geometry (filled polygon)
#   - Bottom panel: computed Bouguer anomaly in mGal
#   - Both panels share the x-axis (horizontal distance in km)
#   - Mark the basin edges with dashed vertical lines

# YOUR CODE HERE

### Question 1.1 — Model sensitivity

Change the basin polygon so the walls are vertical (making it a rectangular
graben rather than a trapezoid).  How does the anomaly shape change?
Specifically: do the edges of the anomaly become sharper or broader?  Why?

**Your response:**

> *(Write 3–5 sentences.)*


---
## Part 2: Linear Inversion *(Supported — ~45 min)*

Given observed gravity data, we want to recover the subsurface density distribution.
We parameterize the subsurface as a grid of rectangular cells, build the
sensitivity matrix $\mathbf{G}$, and solve the regularized inverse problem.

**Available hints:** `p2_G_matrix`, `p2_tikhonov`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Inversion geometry: 20 surface stations, 20 subsurface cells in a single layer.

rng = np.random.default_rng(42)

# Surface stations
x_stations = np.linspace(-9.5e3, 9.5e3, 20)   # m, 20 stations

# Subsurface cells: single layer at depth 1–3 km
cell_width  = 1000.0   # m, each cell is 1 km wide
cell_depth1 = 1000.0   # m, top of cell layer
cell_depth2 = 3000.0   # m, bottom of cell layer
cell_centers_x = np.linspace(-9.5e3, 9.5e3, 20)   # 20 cells
N_cells = len(cell_centers_x)

# True density model (for generating synthetic data)
# A Gaussian density anomaly centered at x = -3 km
true_rho = 300.0 * np.exp(-(cell_centers_x + 3e3)**2 / (2*(2e3)**2))   # kg/m^3

# Helper: rectangle polygon for cell j
def cell_polygon(xc, width, z1, z2):
    """Return (poly_x, poly_z) for a rectangular cell, clockwise."""
    hw = width / 2
    return (np.array([xc-hw, xc+hw, xc+hw, xc-hw]),
            np.array([z1,     z1,    z2,    z2   ]))

print(f'Inversion: {len(x_stations)} stations, {N_cells} cells')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Build the sensitivity matrix G.

def build_sensitivity_matrix(x_stations, cell_centers_x, cell_width, z1, z2):
    """
    Build the sensitivity matrix G for a grid of rectangular prisms.

    Parameters
    ----------
    x_stations : ndarray, shape (M,)
        Surface station positions, meters.
    cell_centers_x : ndarray, shape (N,)
        Horizontal centers of subsurface cells, meters.
    cell_width : float
        Width of each cell, meters.
    z1, z2 : float
        Top and bottom depths of the cell layer, meters (positive down).

    Returns
    -------
    G : ndarray, shape (M, N)
        G[i, j] = gz at station i from cell j with unit density contrast (kg/m^3).
        Units: m/s^2 per kg/m^3.

    Notes
    -----
    Use talwani_gz with delta_rho=1.0 for each cell polygon.
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

G = build_sensitivity_matrix(x_stations, cell_centers_x, cell_width, cell_depth1, cell_depth2)
_check('G shape', G.shape, (20, 20), atol=0)

# Each column of G should be symmetric about the cell center
_col10 = G[:, 10]   # central cell
# The profile is symmetric about the center station, so G[:, 10] should be roughly symmetric
# (not exact because station/cell grids may not align perfectly — check peak is at center)
_peak_idx = np.argmax(_col10)
if abs(_peak_idx - 10) <= 1:
    print(f'  ✓ PASS  Central cell peak near center station (peak at idx {_peak_idx})')
else:
    print(f'  ✗ FAIL  Central cell peak at idx {_peak_idx}, expected ~10')

# G should be positive everywhere (density contrast and gz both positive downward)
if np.all(G >= 0):
    print('  ✓ PASS  G is non-negative (expected for cells below surface)')
else:
    print('  ✗ FAIL  G has negative entries')

# Consistency: G @ true_rho should roughly match forward model
_d_from_G = G @ true_rho
_d_forward = np.array([talwani_gz(np.array([xs]),
                *cell_polygon(xc, cell_width, cell_depth1, cell_depth2),
                true_rho[j])[0]
              for xs in x_stations
              for j, xc in enumerate(cell_centers_x)]).reshape(20, 20).sum(axis=1)  # sum cells
# Simpler check: just verify G @ ones is plausible
print(f'  G @ ones peak = {(G @ np.ones(N_cells)).max()*1e5:.2f} mGal  (should be ~1–10 mGal)')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Tikhonov-regularized inversion.

def tikhonov_inversion(G, d, lam):
    """
    Solve the Tikhonov-regularized least-squares problem.

    Parameters
    ----------
    G : ndarray, shape (M, N)
        Sensitivity matrix.
    d : ndarray, shape (M,)
        Data vector (observed gravity).
    lam : float
        Regularization parameter (lambda).  lam=0 gives pure least-squares.

    Returns
    -------
    m_hat : ndarray, shape (N,)
        Recovered model vector.

    Notes
    -----
    Normal equations:  (G^T G + lam^2 I) m = G^T d
    Use scipy.linalg.solve for numerical stability.
    """
    raise NotImplementedError

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Generate synthetic data from the true model, add noise, then invert.
#
# 1. Compute d_true = G @ true_rho
# 2. Add Gaussian noise: sigma = 0.02 * max(|d_true|)
# 3. Invert for lambda = 0, 0.001, 0.01, 0.1  (in units of mGal / [kg/m^3])
# 4. Plot four panels:
#    - True model and recovered models vs. cell position (km)
#    - Data fit (observed vs. predicted) for each lambda
#
# Which lambda gives the best trade-off between data fit and model smoothness?

noise_sigma = None   # set this after computing d_true

# YOUR CODE HERE

### Question 2.1 — The regularization trade-off

Describe what happens to the recovered density model as $\lambda$ increases
from 0 to large values.  What pathology does $\lambda = 0$ produce?  Why?
What does a very large $\lambda$ do?  Explain in terms of the condition number
of $\mathbf{G}$ and what "minimizing $\|\mathbf{m}\|$" means physically.

**Your response:**

> *(Write 5–7 sentences.)*


### Question 2.2 — Non-uniqueness

Even with the best choice of $\lambda$, your recovered model likely does not
exactly match `true_rho`.  Is this a failure of the algorithm, or is it
expected?  What is the **null space** of $\mathbf{G}$, and why does it prevent
a unique solution?  What type of additional information could reduce the ambiguity?

**Your response:**

> *(Write 4–6 sentences.)*


---
## Part 3: Euler Deconvolution *(Open — ~40 min)*

Euler deconvolution is a fast depth-estimation tool that requires no assumed
source geometry beyond the **structural index** $N$.
You will implement it for a 1D total-field magnetic anomaly and assess how
sensitive the depth estimate is to the choice of $N$.

**Available hint:** `p3_euler_setup`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic total-field magnetic anomaly from Lab 3's dipole geometry.
# True source: point dipole at x=0, depth=3 km.

mu0 = 4 * np.pi * 1e-7

x_euler = np.linspace(-15e3, 15e3, 301)   # m
_true_depth_euler = 3000.0   # m
_I_deg_euler      = 60.0     # regional field inclination
_m_moment_euler   = 5e12     # A·m²

def _dipole_total_field_1d(x, depth, m, I_deg):
    """Total-field anomaly of a point dipole (from Lab 3 geometry)."""
    I = np.radians(I_deg)
    mx, mz = np.cos(I), -np.sin(I)       # induced magnetization direction
    rx, rz = x, depth
    r2 = rx**2 + rz**2
    r  = np.sqrt(r2)
    mr_dot = mx*rx + mz*rz
    pref   = (mu0/(4*np.pi)) * m / r**5
    dBx = pref * (3*mr_dot*rx - mx*r2)
    dBz = pref * (3*mr_dot*rz - mz*r2)
    dF  = dBx*np.cos(I) + dBz*(-np.sin(I))
    return dF * 1e9   # nT

rng2 = np.random.default_rng(7)
F_clean = _dipole_total_field_1d(x_euler, _true_depth_euler, _m_moment_euler, _I_deg_euler)
F_noisy = F_clean + rng2.normal(0, 0.5, size=x_euler.shape)   # 0.5 nT noise

print(f'True source depth: {_true_depth_euler/1000:.1f} km')
print(f'Anomaly peak: {F_noisy.max():.1f} nT, noise sigma: 0.5 nT')

### Task 3.1 — Implement and apply Euler deconvolution

Implement a 1D sliding-window Euler deconvolution.  For each window of width
$W$, set up the $W \times 3$ linear system and solve for $(x_0, z_0, B)$
where $B$ is an unknown background term.  Collect all solutions and filter
to those where the depth estimate is positive and physically reasonable
($0 < z_0 < 10$ km).

Run the deconvolution for structural indices $N = 1$ (dyke/cylinder) and $N = 3$
(sphere/dipole).  Compare the depth estimates to the true depth of 3 km.

Window width: try $W = 20$, 40, 80 points.  How does it affect scatter
and bias in the depth estimate?


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Implement Euler deconvolution and plot results.
#
# Suggested plot: x vs estimated depth for both N values,
# with the true depth as a horizontal dashed line.

# Compute numerical derivatives
dFdx = np.gradient(F_noisy, x_euler)   # nT/m

# For the vertical derivative, use the relationship:
# In the Fourier domain, dF/dz = |k| * F (for upward field from below)
# A simple approximation: compute from the Hilbert transform.
# Or: use a finite difference on the upward-continued field.
# Here we provide the exact vertical derivative computed from the clean signal:
dx = x_euler[1] - x_euler[0]
dFdz_exact = _dipole_total_field_1d(x_euler, _true_depth_euler, _m_moment_euler, _I_deg_euler + 90) * 0  # placeholder

# For your implementation, compute dF/dz using the Hilbert transform relationship:
# dF/dz = H{dF/dx} where H is the Hilbert transform (scipy.signal.hilbert)
# Note: this is exact for a potential field in 2D.
from scipy.signal import hilbert
# dFdz = np.imag(hilbert(dFdx))   # vertical derivative via Hilbert transform

# YOUR CODE HERE

### Question 3.1 — Structural index sensitivity

Compare the depth estimates from $N = 1$ and $N = 3$.  Which gives a better
estimate for this source (which is a dipole-like body)?  What would happen
if the source were actually a 2D dyke ($N = 1$ is correct) but you used $N = 3$?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Synthesis

### S1 — Talwani vs. point-source approximation

The Talwani method gives the exact gravity for a 2D polygon, but geophysicists
often approximate buried bodies as spheres or cylinders for quick estimates.
When is the sphere approximation adequate, and when does the full Talwani
calculation matter?  Support your answer with evidence from Part 1.

**Your response:**

> *(Write 4–5 sentences.)*

### S2 — From forward to inverse

Parts 1 and 2 represent the two directions of the same problem.  Explain how
the forward model (Talwani) becomes the engine for the inversion (sensitivity
matrix).  What assumption was required to make the problem linear, and how
would you handle a **nonlinear** problem (e.g., recovering the geometry of the
basin rather than the density)?

**Your response:**

> *(Write 4–6 sentences.)*

### S3 — Practical depth estimation

Euler deconvolution (Part 3) is used routinely in industry, but it has known
limitations.  List three specific conditions under which Euler deconvolution
would give unreliable depth estimates, and explain physically why each one
causes problems.

**Your response:**

> *(Write 4–6 sentences or a brief numbered list.)*


---
## Extensions *(optional)*

### E1 — L-curve for regularization selection

The L-curve method chooses $\lambda$ by plotting $\|\mathbf{G}\hat{\mathbf{m}} - \mathbf{d}\|$
vs. $\|\hat{\mathbf{m}}\|$ on a log-log scale and finding the corner.
Implement this for the inversion in Part 2 and compare the optimal $\lambda$
to your visually chosen value.

### E2 — Depth weighting

Unregularized gravity inversion tends to pile all the density near the surface
(shallow sources produce larger $G$ entries).  Implement Li & Oldenburg (1998)
depth weighting: multiply each column of $G$ by $1/z_j^{1/2}$ before inversion,
then scale the result back.  How does the recovered model change?

### E3 — 2D inversion grid

Extend Part 2 to a 2D grid of cells (multiple depth layers).  How does the
non-uniqueness problem become more severe?  What is the rank of $\mathbf{G}$
relative to the number of model parameters?


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE